In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomCrop(32, padding = 2),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.Normalize(mean=[0.4914, 0.4821, 0.4465],
                         std=[0.2471, 0.2435, 0.2616])
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4821, 0.4465],
                         std=[0.2471, 0.2435, 0.2616])
])
train_set = datasets.CIFAR10(
    transform = train_transform,
    download = True,
    root = 'data',
    train = True
)
test_set = datasets.CIFAR10(
    transform = test_transform,
    download = True,
    train = False,
    root = 'data'
)
trainloader = DataLoader(
    train_set,
    shuffle = True,
    batch_size = 256,
    num_workers = 4
)
testloader = DataLoader(
    test_set,
    shuffle = True,
    batch_size = 256,
    num_workers = 4
)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super(CNNModel, self).__init__()
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(3, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )

        self.skip1 = nn.Sequential(
            nn.Conv2d(3, 128, 3, stride = 2, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )

        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )
        self.conv_layer4 = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.skip2 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride = 2, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )


        self.conv_layer5 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(512)
        )
        self.conv_layer6 = nn.Sequential(
            nn.Conv2d(512, 512, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(512)
        )
        self.conv_layer7 = nn.Sequential(
            nn.Conv2d(512, 512, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.skip3 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride = 2, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(512)
        )

        self.conv_layer8 = nn.Sequential(
            nn.Conv2d(512, 1024, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(1024)
        )
        self.conv_layer9 = nn.Sequential(
            nn.Conv2d(1024, 1024, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(1024),
        )
        self.conv_layer10 = nn.Sequential(
            nn.Conv2d(1024, 1024, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(1024),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.skip4 = nn.Sequential(
            nn.Conv2d(512, 1024, 3, stride = 2, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(1024)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024*2*2, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)
        )
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode = 'fan_in', nonlinearity = 'relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        skip = self.skip1(x)
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = x + skip

        skip = self.skip2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = x + skip

        skip = self.skip3(x)
        x = self.conv_layer5(x)
        x = self.conv_layer6(x)
        x = self.conv_layer7(x)

        x = x + skip

        skip = self.skip4(x)
        x = self.conv_layer8(x)
        x = self.conv_layer9(x)
        x = self.conv_layer10(x)
        x = x + skip

        out = self.classifier(x)
        return out

In [ ]:
model = CNNModel(num_classes = 10, dropout_rate = 0.3)
model.to(device)
print(model)

CNNModel(
  (conv_layer1): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer2): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.3, inplace=False)
  )
  (skip1): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True,

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 5e-4)
decayRate = 0.96
learning_rate_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer = optimizer, gamma = decayRate)

In [ ]:
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

In [ ]:
def evaluate(model, criterion, trainloader):
    model.eval()
    test_loss = 0.0
    test_accuracy = 0
    total = 0
    with torch.no_grad():
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * labels.size(0)

            predicted = torch.argmax(outputs.data, dim = 1)

            test_accuracy += (predicted == labels).sum().item()

            total += labels.size(0)
    test_loss /= total
    test_accuracy = test_accuracy * 100 / total
    return test_loss, test_accuracy

In [ ]:
epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_losses = 0.0
    epoch_accuracies = 0
    total = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        epoch_losses += loss.item() * labels.size(0)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
        total += labels.size(0)
        predicted = torch.argmax(outputs.data, dim = 1)

        epoch_accuracies += (predicted == labels).sum().item()
    learning_rate_scheduler.step()
    epoch_losses /= total
    epoch_accuracies = epoch_accuracies * 100 / total
    test_loss, test_accuracy = evaluate(model, criterion, testloader)

    print(
        f'Epoch: [{epoch + 1}/{epochs}] | ',
        f'Train Loss: {epoch_losses:.4f} | ',
        f'Test Loss: {test_loss:.4f} | ',
        f'Train Accuracy: {epoch_accuracies:.2f}% | ',
        f'Test Accuracy: {test_accuracy:.2f}% | '
    )
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)
    train_losses.append(epoch_losses)
    train_accuracies.append(epoch_accuracies)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_losses, label = 'Train Losses')
plt.plot(test_losses, label = 'Test Losses')
plt.legend()

In [ ]:
plt.plot(train_accuracies, label = 'Train Accuracies')
plt.plot(test_accuracies, label = 'Test Accuracies')